In [2]:
import fastf1
import pandas as pd
from rapidfuzz.process_cpp_impl import INT64

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

fastf1.Cache.enable_cache("../ff1_cache")

session = fastf1.get_session(2023, "Spanish Grand Prix", "R")
session.load(laps=True, telemetry=False, weather=False, messages=False)

laps = session.laps

core           INFO 	Loading data for Spanish Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 1 completed the race distance 00:00.037000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '44', '63', '11', '55', '18', '14', '31', '24', '10', '16', '22', '81', '21', '27', '23', '4', '20', '77', '2']


In [3]:
laps["Compound"].unique()

array(['MEDIUM', 'HARD', 'SOFT'], dtype=object)

In [4]:
dim_compound = pd.DataFrame({
    "compound_key": [1, 2, 3, 4, 5],
    "compound_name": ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"],
})
dim_compound

,compound_key,compound_name
0,1,SOFT
1,2,MEDIUM
2,3,HARD
3,4,INTERMEDIATE
4,5,WET


In [5]:
session.results[["Abbreviation", "FullName", "DriverNumber", "TeamName"]]

,Abbreviation,FullName,DriverNumber,TeamName
1,VER,Max Verstappen,1,Red Bull Racing
44,HAM,Lewis Hamilton,44,Mercedes
63,RUS,George Russell,63,Mercedes
11,PER,Sergio Perez,11,Red Bull Racing
55,SAI,Carlos Sainz,55,Ferrari
18,STR,Lance Stroll,18,Aston Martin
14,ALO,Fernando Alonso,14,Aston Martin
31,OCO,Esteban Ocon,31,Alpine
24,ZHO,Guanyu Zhou,24,Alfa Romeo
10,GAS,Pierre Gasly,10,Alpine


In [6]:
dim_driver = (
    session.results[["Abbreviation", "FullName", "DriverNumber"]]
    .reset_index(drop=True)
    .rename(columns={
        "Abbreviation": "driver_code",
        "FullName": "driver_name",
        "DriverNumber": "driver_number",
    })
)
dim_driver.insert(0, "driver_key", range(1, len(dim_driver) + 1))
dim_driver

,driver_key,driver_code,driver_name,driver_number
0,1,VER,Max Verstappen,1
1,2,HAM,Lewis Hamilton,44
2,3,RUS,George Russell,63
3,4,PER,Sergio Perez,11
4,5,SAI,Carlos Sainz,55
5,6,STR,Lance Stroll,18
6,7,ALO,Fernando Alonso,14
7,8,OCO,Esteban Ocon,31
8,9,ZHO,Guanyu Zhou,24
9,10,GAS,Pierre Gasly,10


In [7]:
dim_team = (
    session.results[["TeamName"]].drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={
        "TeamName": "team_name",
    })
)
dim_team.insert(0, "team_key", range(1, len(dim_team) + 1))
dim_team

,team_key,team_name
0,1,Red Bull Racing
1,2,Mercedes
2,3,Ferrari
3,4,Aston Martin
4,5,Alpine
5,6,Alfa Romeo
6,7,AlphaTauri
7,8,McLaren
8,9,Haas F1 Team
9,10,Williams


In [8]:
dim_session = pd.DataFrame({
    "session_key": [1] ,
    "year": [2023] ,
    "round_number": [session.event["RoundNumber"]] ,
    "event_name": [session.event["EventName"]] ,
    "country": [session.event["Country"]] ,
    "location": [session.event["Location"]] ,
    "event_format": [session.event["EventFormat"]] ,
    "session_type": ["R"] ,
})
dim_session

,session_key,year,round_number,event_name,country,location,event_format,session_type
0,1,2023,7,Spanish Grand Prix,Spain,Barcelona,conventional,R


In [9]:
laps.merge(dim_team, left_on="Team", right_on="team_name", how="left")

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,team_key,team_name
0,0 days 01:03:40.107000,VER,1,0 days 00:01:23.935000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:32.084000,0 days 00:00:23.926000,NaT,0 days 01:03:16.243000,0 days 01:03:40.243000,256.0,261.0,276.0,275.0,False,MEDIUM,1.0,True,Red Bull Racing,0 days 01:02:15.963000,NaT,1,1.0,None,,False,False,1,Red Bull Racing
1,0 days 01:05:00.509000,VER,1,0 days 00:01:20.402000,2.0,1.0,NaT,NaT,0 days 00:00:24.186000,0 days 00:00:32.088000,0 days 00:00:24.128000,0 days 01:04:04.338000,0 days 01:04:36.426000,0 days 01:05:00.554000,252.0,257.0,276.0,295.0,True,MEDIUM,2.0,True,Red Bull Racing,0 days 01:03:40.107000,NaT,1,1.0,None,,False,True,1,Red Bull Racing
2,0 days 01:06:21.008000,VER,1,0 days 00:01:20.499000,3.0,1.0,NaT,NaT,0 days 00:00:24.167000,0 days 00:00:32.191000,0 days 00:00:24.141000,0 days 01:05:24.721000,0 days 01:05:56.912000,0 days 01:06:21.053000,249.0,256.0,276.0,297.0,False,MEDIUM,3.0,True,Red Bull Racing,0 days 01:05:00.509000,NaT,1,1.0,None,,False,True,1,Red Bull Racing
3,0 days 01:07:41.354000,VER,1,0 days 00:01:20.346000,4.0,1.0,NaT,NaT,0 days 00:00:24.022000,0 days 00:00:32.159000,0 days 00:00:24.165000,0 days 01:06:45.075000,0 days 01:07:17.234000,0 days 01:07:41.399000,255.0,256.0,276.0,300.0,True,MEDIUM,4.0,True,Red Bull Racing,0 days 01:06:21.008000,NaT,1,1.0,None,,False,True,1,Red Bull Racing
4,0 days 01:09:01.637000,VER,1,0 days 00:01:20.283000,5.0,1.0,NaT,NaT,0 days 00:00:24.034000,0 days 00:00:32.213000,0 days 00:00:24.036000,0 days 01:08:05.433000,0 days 01:08:37.646000,0 days 01:09:01.682000,254.0,256.0,277.0,301.0,True,MEDIUM,5.0,True,Red Bull Racing,0 days 01:07:41.354000,NaT,1,1.0,None,,False,True,1,Red Bull Racing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1307,0 days 02:25:43.001000,SAR,2,0 days 00:01:21.280000,61.0,3.0,NaT,NaT,0 days 00:00:24.354000,0 days 00:00:32.587000,0 days 00:00:24.339000,0 days 02:24:46.114000,0 days 02:25:18.701000,0 days 02:25:43.040000,265.0,265.0,280.0,306.0,False,HARD,25.0,True,Williams,0 days 02:24:21.721000,NaT,1,20.0,None,,False,True,10,Williams
1308,0 days 02:27:05.135000,SAR,2,0 days 00:01:22.134000,62.0,3.0,NaT,NaT,0 days 00:00:23.675000,0 days 00:00:33.473000,0 days 00:00:24.986000,0 days 02:26:06.715000,0 days 02:26:40.188000,0 days 02:27:05.174000,271.0,192.0,280.0,308.0,False,HARD,26.0,True,Williams,0 days 02:25:43.001000,NaT,1,20.0,None,,False,True,10,Williams
1309,0 days 02:28:25.555000,SAR,2,0 days 00:01:20.420000,63.0,3.0,NaT,NaT,0 days 00:00:23.634000,0 days 00:00:32.486000,0 days 00:00:24.300000,0 days 02:27:28.808000,0 days 02:28:01.294000,0 days 02:28:25.594000,264.0,273.0,280.0,310.0,False,HARD,27.0,True,Williams,0 days 02:27:05.135000,NaT,1,20.0,None,,False,True,10,Williams
1310,0 days 02:29:45.535000,SAR,2,0 days 00:01:19.980000,64.0,3.0,NaT,NaT,0 days 00:00:23.602000,0 days 00:00:32.127000,0 days 00:00:24.251000,0 days 02:28:49.196000,0 days 02:29:21.323000,0 days 02:29:45.574000,279.0,278.0,280.0,308.0,False,HARD,28.0,True,Williams,0 days 02:28:25.555000,NaT,1,20.0,None,,False,True,10,Williams


In [34]:
fct = laps.merge(dim_team, left_on="Team", right_on="team_name", how="left")
fct = fct.merge(dim_driver, left_on="Driver", right_on="driver_code", how="left")
fct = fct.merge(dim_compound, left_on="Compound", right_on="compound_name", how="left")

fct["session_key"] = 1

fct = fct.drop(columns=["Driver", "DriverNumber", "Team", "Compound", "team_name", "driver_code", "driver_name", "driver_number", "compound_name", "FastF1Generated"])

fct["is_first_lap"] = fct["LapNumber"] == 1
fct["is_in_lap"] = fct["PitInTime"].notna()
fct["is_out_lap"] = fct["PitOutTime"].notna()
fct["is_pace_lap"] = ~(fct["is_first_lap"] | fct["is_in_lap"] | fct["is_out_lap"])

fct = fct.astype({
    "LapNumber": "Int64",
    "Stint": "Int64",
    "TyreLife": "Int64",
    "Position": "Int64"
})

fct = fct.rename(columns={"Time": "time",
                    "LapTime": "lap_time",
                    "LapNumber": "lap_number",
                    "Stint": "stint",
                    "PitOutTime": "pit_out_time",
                    "PitInTime": "pit_in_time",
                    "Sector1Time": "sector1_time",
                    "Sector2Time": "sector2_time",
                    "Sector3Time": "sector3_time",
                    "SpeedI1": "speed_i1",
                    "SpeedI2": "speed_i2",
                    "SpeedFL": "speed_fl",
                    "SpeedST": "speed_st",
                    "Sector1SessionTime": "sector1_session_time",
                    "Sector2SessionTime": "sector2_session_time",
                    "Sector3SessionTime": "sector3_session_time",
                    "IsPersonalBest": "is_personal_best",
                    "TyreLife": "tyre_life",
                    "FreshTyre": "fresh_tyre",
                    "LapStartTime": "lap_start_time",
                    "LapStartDate": "lap_start_date",
                    "TrackStatus": "track_status",
                    "Position": "position",
                    "Deleted": "deleted",
                    "DeletedReason": "deleted_reason",
                    "IsAccurate": "is_accurate",

                    })

print(fct.columns.tolist())
print(fct["is_pace_lap"].sum())
print(len(fct))
print(fct[["lap_number", "stint", "tyre_life", "position"]].dtypes)
fct[["team_key", "driver_key", "compound_key"]].isna().sum()

['time', 'lap_time', 'lap_number', 'stint', 'pit_out_time', 'pit_in_time', 'sector1_time', 'sector2_time', 'sector3_time', 'sector1_session_time', 'sector2_session_time', 'sector3_session_time', 'speed_i1', 'speed_i2', 'speed_fl', 'speed_st', 'is_personal_best', 'tyre_life', 'fresh_tyre', 'lap_start_time', 'lap_start_date', 'track_status', 'position', 'deleted', 'deleted_reason', 'is_accurate', 'team_key', 'driver_key', 'compound_key', 'session_key', 'is_first_lap', 'is_in_lap', 'is_out_lap', 'is_pace_lap']
1207
1312
lap_number    Int64
stint         Int64
tyre_life     Int64
position      Int64
dtype: object


team_key        0
driver_key      0
compound_key    0
dtype: int64